# Key Metric Comparison

This notebook allows the comparison of two prototype iterations based on specific key metrics.

In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
from IPython.display import display

In [ ]:
pd.options.plotting.backend = "plotly"

baseline_runs = [
    "", "", "" # enter results files to analyse
]

comparing_runs = [
    "", "", "" # enter results files to analyse
]

metrics = [
    "context_precision","context_recall","nv_context_relevance","answer_relevancy",
    "nv_response_groundedness", "correctness","semantic_similarity",
    "trustworthiness","prompt_alignment"
]

metrics_to_compare = [ # choose subset of metrics to compare
    "context_precision","context_recall","nv_context_relevance","answer_relevancy",
    "nv_response_groundedness", "correctness","semantic_similarity",
    "trustworthiness","prompt_alignment"
]

Flatten and combine all results from all runs into one dataframe

In [ ]:
baseline_data = []

for run_file in baseline_runs:
    with open(run_file, 'r') as f:
        run_data = json.load(f)
        run_name = f.name.split('/')[-1]
        for entry in run_data['results']:
            dataset_name = entry["dataset"]
            
            for result in entry["results"]:
                r = result.copy()
                r["dataset"] = dataset_name
                r["run"] = run_name
                baseline_data.append(r)

baseline_df = pd.DataFrame(baseline_data)

baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'context_precision'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'context_recall'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'answer_relevancy'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'semantic_similarity'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'trustworthiness'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'nv_response_groundedness'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'multiple choice', 'answer_relevancy'] = float('nan')

In [ ]:
comparing_data = []

for run_file in comparing_runs:
    with open(run_file, 'r') as f:
        run_data = json.load(f)
        run_name = f.name.split('/')[-1]
        for entry in run_data['results']:
            dataset_name = entry["dataset"]
            
            for result in entry["results"]:
                r = result.copy()
                r["dataset"] = dataset_name
                r["run"] = run_name
                comparing_data.append(r)

comparing_df = pd.DataFrame(comparing_data)

comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'context_precision'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'context_recall'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'answer_relevancy'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'semantic_similarity'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'trustworthiness'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'nv_response_groundedness'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'multiple choice', 'answer_relevancy'] = float('nan')

In [ ]:
baseline_detail = baseline_df.groupby('dataset')[metrics_to_compare].mean()
baseline_by_metric = baseline_df[metrics_to_compare].mean()

comparing_detail = comparing_df.groupby('dataset')[metrics_to_compare].mean()
comparing_by_metric = comparing_df[metrics_to_compare].mean()

baseline_by_dataset = baseline_df.groupby('dataset')[metrics_to_compare].mean().T.mean()
comparing_by_dataset = comparing_df.groupby('dataset')[metrics_to_compare].mean().T.mean()

baseline_by_run = baseline_df.groupby('run')[metrics_to_compare].mean()
comparing_by_run = comparing_df.groupby('run')[metrics_to_compare].mean()

## Visualizing results

In [ ]:
baseline_overall = baseline_df[metrics].mean().mean()
comparing_overall = comparing_df[metrics].mean().mean()

fig = go.Figure(go.Indicator(
    mode = "number+delta",
    value = np.round(comparing_overall,3),
    number = {'prefix': f"Overall changed from {np.round(baseline_overall,3)} to "},
    delta = {'position': "top", 'reference': np.round(baseline_overall,3)}
))

fig.show()

In [ ]:
fig = go.Figure()

baseline_text = np.round(baseline_by_metric.values.tolist(), 2)
comparing_text = np.round(comparing_by_metric.values.tolist(), 2)

fig.add_trace(
    go.Bar(
        x=baseline_by_metric.index.tolist(),
        y=baseline_by_metric.values.tolist(),
        name="baseline",
        marker_color="grey",
        text=baseline_text
    )
)

fig.add_trace(
    go.Bar(
        x=baseline_by_metric.index.tolist(),
        y=comparing_by_metric.values.tolist(),
        name="comparing",
        marker_color='#0EA4E3',
        text=comparing_text
    )
)

fig.update_layout(
    barmode='group',
    yaxis=dict(
        range=[0, 1],
        title="Mean score"
    ),
    xaxis=dict(
        title="Metric"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

In [ ]:
fig = go.Figure()

baseline_text = np.round(baseline_by_dataset.values.tolist(), 2)
comparing_text = np.round(comparing_by_dataset.values.tolist(), 2)

fig.add_trace(
    go.Bar(
        x=baseline_by_dataset.index.tolist(),
        y=baseline_by_dataset.values.tolist(),
        name="baseline",
        marker_color="grey",
        text=baseline_text
    )
)

fig.add_trace(
    go.Bar(
        x=baseline_by_dataset.index.tolist(),
        y=comparing_by_dataset.values.tolist(),
        name="comparing",
        marker_color='#0EA4E3',
        text=comparing_text
    )
)


fig.update_layout(
    barmode='group',
    yaxis=dict(
        range=[0, 1],
        title="Mean score"
    ),
    xaxis=dict(
        title="Metric"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

In [ ]:
metrics_diffs = comparing_by_metric - baseline_by_metric
metrics_percentage_diff = np.round(metrics_diffs / baseline_by_metric * 100, 2)

fig = go.Figure()

for metric in metrics_diffs.index:
    y = metrics_diffs[metric]
    color = "#FF7B52" if y <= 0.0 else "#56D752"
    fig.add_trace(
        go.Bar(x=[metric], y=[y], marker_color=color, text=f"{np.round(y, 4)}<br>({metrics_percentage_diff[metric]}%)")
    )

fig.update_layout(
    yaxis=dict(
        range=[-0.4, 0.4],
        title="Delta"
    ),
    xaxis=dict(
        title="Considered metrics"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

In [ ]:
dataset_diffs = comparing_by_dataset - baseline_by_dataset
dataset_percentage_diffs = np.round(dataset_diffs / baseline_by_dataset * 100, 2)

fig = go.Figure()

for dataset in dataset_diffs.index:
    y = dataset_diffs[dataset]
    color = "#FF7B52" if y <= 0.0 else "#56D752"
    fig.add_trace(
        go.Bar(x=[dataset], y=[y], marker_color=color, text=f"{np.round(y, 4)}<br>({dataset_percentage_diffs[dataset]}%)")
    )

fig.update_layout(
    yaxis=dict(
        range=[-0.4, 0.4],
        title="Delta"
    ),
    xaxis=dict(
        title="Dataset"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()